# Analysis of Drinking Waterpoints in Kano, Nigeria
> Note: This notebook requires the local environment dependencies listed in our [requirements.txt] (requirements.txt) file. Use this file to install the required packages in a virtual environment.

> To excecute OpenRouteService functions, it is required to install the [library dependencies](https://github.com/GIScience/openrouteservice-examples#local-installation). You should either have an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local ORS environment to complete the analysis.

The model concepts and processes are described in our documentation. The [Dataset-interpretability](https://github.com/urbanbigdatacentre/ideamaps-models/blob/a4084fb650424ac575941cdacb71421aa882bae4/models/emergency-maternal-care/kano/dataset-interpretability.md) file describes the rationale behind this model.

## Workflow:
The notebook is divided into the following sections:

1. Initial Setup
2. Data Preparation
3. Travel time estimates
4. Two-step floating catchment area (2SFCA) analysis
5. Results

## 1. Initial Setup

## Setting up the virtual environment

```bash
# Create a new virtual environment
# It is recommended to create this virtual environment in the scripts folder
python -m venv .venv

# Activate the virtual environment
source .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [ ]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

## Preprocessing
In this study, users first requested an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.

### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# %%
# Read the api key from the .env file
from dotenv import load_dotenv
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
    # Insert R code from the local ORS service
```

For the drinking‑water model in Kano State, we began with 684 waterpoint records from the [Donate Water initiative](https://donatewater.ng), a community‑driven platform that combines crowdsourcing and data analytics to map and monitor water facilities across Nigeria. In [QGIS](https://qgis.org/), we removed six points located more than 50 km from their nearest neighbours, yielding 678 non‑outlier records. We then applied a second round of quality‑control filters—excluding records with “I don’t know” values for water source, ownership, or cost, as well as facilities listed as abandoned or non‑functional—to arrive at our final, validated dataset of 456 waterpoints.

Boundary Framework
When these 678 non‑outlier points were overlaid on the official Kano Local Government Area (Admin Level 2) boundary [Humanitarian Data Exchange, published 25 Nov 2015](https://data.humdata.org/) , 56 (8.3 %) lay just outside the jurisdictional line, clustering along major roads. To ensure our coverage metrics reflected the true operational footprint, we hand‑digitized a bespoke polygon in QGIS that fully encloses all 678 non‑outlier points. After final cleaning, all 456 validated waterpoints fall within this custom service‑area boundary.

Usage in Analysis
From that point onward, every spatial query, point‑density calculation, and service‑area model was executed strictly within our hand‑digitized boundary. By anchoring the analysis to the actual distribution of cleaned, functional waterpoints, we ensure that our density and coverage metrics faithfully represent on‑the‑ground service provision—avoiding the underestimation that would result from forcing the data into administrative lines misaligned with real‑world facility locations.

Data Sources
* [Waterpoint facilities](https://donatewater.ng): generated 12 Oct 2023; accessed 15 Feb 2025 
* [District boundaries](https://data.humdata.org/dataset/nigeria-admin-level-2): Humanitarian Data Exchange, published 25 Nov 2015; accessed 10 Jan 2024
* [World Population Data](https://hub.worldpop.org/geodata/summary?id=28031): generated 2020-02-01; accessed 15 july 2025
* Custom service‑area boundary: Hand‑digitized in QGIS to encompass all non‑outlier waterpoints (2025)

In [ ]:
# Set paths to access Kano data
# Define directories
data_inputs = '../scripts/Kano/data-inputs/'
data_temp = '../scripts/Kano/data-temp/'
model_outputs = '../kano/'

## Data Collection

### 1.1 Drinking waterpoints for Kano

In [ ]:
# Read the drinking water points data
# Ensure the file path is correct and the file exists
drinking_waterpoints = gpd.read_file(data_inputs + 'waterpoints_kano.geojson')

In [ ]:
drinking_waterpoints

In [ ]:
# Define unwanted categories
unwanted_sources = ["No, it is not functional and not in use", "No, it is abandoned", "I don't know"]

# Filter them out
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterSourc'].isin(unwanted_sources)]


In [ ]:
# Map the water source valuess to better descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterSourc'] = (
    drinking_waterpoints['waterSourc']
    .astype(str)
    .str.strip()
    .map({
        'Yes, it is fully functional and still in use': 'Fully Functional',
        'Yes, it is partially functional and still in use': 'Partially Functional'
    })
)

In [ ]:
# Define unwanted categories
unwanted_sources1 = ["Don't know"]

# Filter them out
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterPubli'].isin(unwanted_sources1)]
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterFree'].isin(unwanted_sources1)]


In [ ]:
drinking_waterpoints

In [ ]:
# Define mapping from waterState to waterDrink for "I don't know"
state_to_drink = {
    "Clear": "Yes",
    "Dirty": "No",
    "Currently no water available": "No",
    "Muddy": "No",
    "Coloured": "No"
}

# Copy the original waterDrink column
drinking_waterpoints['waterDrink_estimated'] = drinking_waterpoints['waterDrink']

# Only update values in waterDrink_estimated where waterDrink is "I don't know"
drinking_waterpoints.loc[drinking_waterpoints['waterDrink'] == "Don't know", 'waterDrink_estimated'] = drinking_waterpoints.loc[
    drinking_waterpoints['waterDrink'] == "Don't know", 'waterState'
].map(state_to_drink)

In [ ]:
drinking_waterpoints

In [ ]:
# Define user-specified improved water source types
improved = [
    'piped-borne water',
    'borehole with hand pump',
    'borehole with tap',
    'water vendors',
    'spring'
]

# Create a new column 'watertype_revised' with classification
drinking_waterpoints['watertype_revised'] = drinking_waterpoints['waterType'].apply(
    lambda x: 'Improved' if isinstance(x, str) and x.strip().lower() in improved else 'Unimproved'
)

drinking_waterpoints

In [ ]:
# map the waterDrink_estimated values to more descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterDrink_estimated'] = (
    drinking_waterpoints['waterDrink_estimated']
    .astype(str)
    .str.strip()
    .map({
        'Yes': 'Drinkable',
        'No': 'Non drinkable'
    })
)

In [ ]:
# map the waterFree values to more descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterFree'] = (
    drinking_waterpoints['waterFree']
    .astype(str)
    .str.strip()
    .map({
        'Yes': 'Free',
        'No': 'Non Free'
    })
)

In [ ]:
drinking_waterpoints

In [ ]:
# 1. Load or reference your two GeoDataFrames
#    – drinking_waterpoints: your 456 waterpoints
#    – study_area: the polygon(s) defining your Kano study area
#      (e.g. from "grid-boundary-kano.gpkg")
study_area = gpd.read_file(data_inputs + "grid-boundary-kano.gpkg")

# 2. Ensure both are in the same CRS
if drinking_waterpoints.crs != study_area.crs:
    drinking_waterpoints = drinking_waterpoints.to_crs(study_area.crs)

drinking_waterpoints = gpd.sjoin(drinking_waterpoints, study_area, how="inner", predicate="intersects")
drinking_waterpoints = drinking_waterpoints[drinking_waterpoints.columns]
drinking_waterpoints


In [ ]:
# Define conditions for categorizing water points into limited, optimal, and moderate water categories
# Ensure the conditions are correctly defined based on the data
conditions = [
    drinking_waterpoints['waterDrink_estimated'] == 'Non drinkable',
    (drinking_waterpoints['waterSourc'] == 'Fully Functional') & (drinking_waterpoints['watertype_revised'] == 'Improved')
]

categories = ['Limited Water', 'Optimal Water']

drinking_waterpoints['category'] = np.select(conditions, categories, default='Moderate Water')

In [ ]:
# Display the counts of each category
# Ensure the counts are correctly calculated
category_counts = drinking_waterpoints['category'].value_counts()
category_counts

In [ ]:
drinking_waterpoints

In [ ]:
# Assign a unique identifier to each water point
# Ensure the identifier is unique and sequential
drinking_waterpoints['waterpoint_id'] = range (1, len(drinking_waterpoints) + 1)

In [ ]:
drinking_waterpoints

In [ ]:
# Write to GeoJSON
drinking_waterpoints.to_file(data_inputs + 'Kano_DW.geojson', driver='GeoJSON')

# Population Grid Data

In [ ]:
study_area = gpd.read_file(data_inputs + "grid-boundary-kano.gpkg")
population_data = data_inputs + "nga_ppp_2020_UNadj.tif"

In [ ]:
study_area["grid_id"] = range(1, len(study_area) + 1)

In [ ]:
with rasterio.open(population_data) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

In [ ]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [ ]:
with rasterio.open(data_inputs + 'kano_nga_ppp_2020_UNadj.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

In [ ]:
# 1. Open the population raster
with rasterio.open(data_inputs + "Kano_nga_ppp_2020_UNadj.tif") as src:
    data = src.read(1)
    transform = src.transform
    crs = src.crs

    rows, cols = data.shape

    polygons = []
    values = []

    for row in range(rows):
        for col in range(cols):
            value = data[row, col]
            if value == src.nodata:
                continue  

            x, y = rasterio.transform.xy(transform, row, col, offset='ul')

            pixel_width = transform.a
            pixel_height = -transform.e 

            poly = box(x, y - pixel_height, x + pixel_width, y)
            polygons.append(poly)
            values.append(value)

population_kano = gpd.GeoDataFrame({
    "value": values,
    "geometry": polygons
}, crs=crs)

# 7. Save to GeoPackage (two layers: polygons and centroids)
population_kano.to_file("population_kano.gpkg", driver="GPKG")

In [ ]:
# 5. Calculate centroids
population_kano["centroid"] = population_kano.geometry.centroid

# 6. Create centroid layer (points)
population_centroids = population_kano[["value", "centroid"]].copy()
population_centroids = population_centroids.rename(columns={"centroid": "geometry"})
population_centroids = population_centroids.set_geometry("geometry")
population_centroids.set_crs("EPSG:4326", inplace=True)

# 4. Add unique ID
population_centroids["id"] = range(1, len(population_centroids) + 1)

population_centroids.to_file("population_centroids_kano.gpkg", driver="GPKG")

In [ ]:

population_centroids

In [ ]:
# === Ensure same CRS ===
if study_area.crs != population_centroids.crs:
    population_centroids = population_centroids.to_crs(study_area.crs)

# === STEP 1: Spatial join to assign population values ===
join_gdf = gpd.sjoin(
    population_centroids,
    study_area,
    how="left",
    predicate="within"
)

# Compute mean population for each grid_id (works for 1 or 2 centroids)
pop_by_grid = (
    join_gdf.groupby("grid_id")
    .agg({"value": "mean"})
    .reset_index()
)

# Merge into grid
kano_grid_with_pop = study_area.merge(pop_by_grid, on="grid_id", how="left")

# 5. Identify empty grids (no population assigned)
empty_grids = kano_grid_with_pop[kano_grid_with_pop['value'].isna()].copy()

if not empty_grids.empty:
    # 6. Build KDTree from population centroid coordinates
    pop_points = np.array([(pt.x, pt.y) for pt in population_centroids.geometry])
    pop_tree = cKDTree(pop_points)

    # 7. Get centroids of empty grids
    empty_centroids = np.array([(geom.centroid.x, geom.centroid.y) for geom in empty_grids.geometry])

    # 8. Query nearest population centroid for each empty grid centroid
    dist, idx = pop_tree.query(empty_centroids, k=1)

    # 9. Retrieve population values of nearest centroids
    nearest_values = population_centroids.iloc[idx]['value'].values

    # 10. Assign these values to the empty grids
    empty_grids['value'] = nearest_values
    kano_grid_with_pop.loc[empty_grids.index, 'value'] = empty_grids['value']

# 11. Save the updated grid with population values to a GeoPackage
output_path = data_temp + "kano_grid_pop.gpkg"
kano_grid_with_pop.to_file(output_path, driver="GPKG")


In [ ]:
kano_grid_pop_centroids = kano_grid_with_pop.copy()
kano_grid_pop_centroids['centroid'] = kano_grid_pop_centroids.geometry.centroid
kano_grid_pop_centroids = kano_grid_pop_centroids.set_geometry('centroid')

kano_grid_pop_centroids = kano_grid_pop_centroids.drop(columns='geometry')

In [ ]:


kano_grid_pop_centroids.to_file(
    data_temp + 'kano_grid_pop_centroids.geojson',
    driver='GeoJSON'
)


# OD matrix

In [ ]:
# Read the Kano OD matrix
# Ensure the file path is correct and the file exists
Kano_OD_Matrix = pd.read_csv(data_inputs + 'kano_odmatrix.csv')
Kano_OD_Matrix

In [ ]:
# Filter rows where duration_seconds <= 3600
Kano_OD_Matrix_quota = Kano_OD_Matrix[Kano_OD_Matrix['duration_seconds'] <= 3600]  # 1 hour

Kano_OD_Matrix_quota

In [ ]:
# Merge the Kano OD matrix with the population centroids grid cells
pop_centroid_matrix_quota = pd.merge(Kano_OD_Matrix_quota, kano_grid_with_pop[['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','value', 'geometry']],
                             left_on='origin_id', right_on='grid_id', how='left')

In [ ]:
# Display the merged population centroid matrix
# Ensure the data is correctly merged and displayed
pop_centroid_matrix_quota

In [ ]:
# Rename columns for clarity
pop_centroid_matrix_quota = pop_centroid_matrix_quota.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "destination_id": "dwp_id",
    "value": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min", "origin_lat_min", "origin_lon_max", "origin_lat_max", "population", "geometry", "dwp_id", "duration_seconds","duration_minutes", "distance_km"]
pop_centroid_matrix_quota = pop_centroid_matrix_quota[columns_to_keep]

In [ ]:
pop_centroid_matrix_quota

In [ ]:
# Convert waterpoint_id to numeric and ensure it is an integer
drinking_waterpoints['waterpoint_id'] = pd.to_numeric(drinking_waterpoints['waterpoint_id'], errors='raise').astype(int)

print(drinking_waterpoints['waterpoint_id'].dtype) 

In [ ]:
print(pop_centroid_matrix_quota['dwp_id'].dtype) 

In [ ]:
# Merge the population centroid matrix with the drinking water points
# Ensure the merge is done correctly and the columns are aligned
# This will add the waterpoint_id and other relevant columns to the population centroid matrix
distances_duration_matrix = pd.merge(pop_centroid_matrix_quota, drinking_waterpoints[['waterpoint_id', 'category', 'userPosi_1', 'userPosi_2']], 
                     left_on='dwp_id', right_on='waterpoint_id', how='left')

In [ ]:
distances_duration_matrix

In [ ]:
distances_duration_matrix = gpd.GeoDataFrame(distances_duration_matrix, geometry="geometry", crs="EPSG:4326")
distances_duration_matrix.to_file(data_temp + 'distances_duration_matrix_quota.geojson', driver='GeoJSON')

In [ ]:
distances_duration_matrix

In [ ]:
# Count the number of water points in each category
# Ensure the counts are correctly calculated
category_counts = drinking_waterpoints['category'].value_counts()
print(category_counts)

In [ ]:
# Display the unique categories in the distances_duration_matrix
print(distances_duration_matrix['category'].unique())

In [ ]:
# Define the categories for water points
categories = {
    'Optimal_Water': ['Optimal Water'],
    'Moderate_Water': ['Moderate Water'],
    'Limited_Water': ['Limited Water']
} 

In [ ]:
# step 1: Create subsets of the distances_duration_matrix based on the defined categories
# Ensure the subsets are correctly created and the data is filtered based on the categories
subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['category'].isin(values)
    ]
    for key, values in categories.items()
}

In [ ]:
# Extract subsets for each category
Optimal_Water_subset = subsets['Optimal_Water']
Moderate_Water_subset = subsets['Moderate_Water']
Limited_Water_subset = subsets['Limited_Water']

In [ ]:
# Step 2: Define a function to get the smallest duration_seconds per grid_id for each category
def get_closest(df, n=1):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_minutes')).reset_index(drop=True)


In [ ]:
# If the subsets are already created for each category, we apply the function to each subset:
Optimal_Water_closest = get_closest(Optimal_Water_subset)
Moderate_Water_closest = get_closest(Moderate_Water_subset)
Limited_Water_closest = get_closest(Limited_Water_subset)

In [ ]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    Optimal_Water_closest, Moderate_Water_closest, Limited_Water_closest])

In [ ]:
distances_duration_matrix

In [ ]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "userPosi_2": "dest_lon",
    "userPosi_1": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['dwp_id'])

In [ ]:
# Add geometry to the DataFrame because the OD matrix is a CSV file and does not have geometry information.
# We create a geometry column using the origin coordinates.
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry="geometry", crs="EPSG:4326")

In [ ]:
# Ensure the geometry is correctly created from the origin coordinates
gpkg_path = data_temp + 'distances_duration_closest_DW.gpkg'
gdf.to_file(gpkg_path, layer="distances_duration_closest_DW", driver="GPKG")

In [ ]:
# Review and remove
origin_dest = gpd.read_file(data_temp + "distances_duration_closest_DW.gpkg", layer="distances_duration_closest_DW")

In [ ]:
origin_dest

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [ ]:
# Function to calculate beta based on the maximum distance and a given W value
from math import *
d = 60 * 60 # quota has been applied to limit the maximum duration to 1 hour
W = 0.01 # try 0.1, 0.05, 0.01, 0.75
beta = - d ** 2 / log(W)
print(beta)

In [ ]:
# Display the first few rows of the origin_dest DataFrame
print(origin_dest.head())

In [ ]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [ ]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')

In [ ]:
origin_dest_acc = origin_dest

In [ ]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact. 
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [ ]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [ ]:
origin_dest_acc

In [ ]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='waterpoint_id')['Pop_W'].sum().reset_index()

In [ ]:
origin_dest_sum

In [ ]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='waterpoint_id')

In [ ]:
origin_dest_acc

In [ ]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [ ]:
supply_map = {
    'Optimal Water': 1.0,   
    'Moderate Water': 0.5,  
    'Limited Water': 0.2    
}

In [ ]:
# Map the supply values to the categories
origin_dest_acc['supply'] = origin_dest_acc['category'].map(supply_map)

In [ ]:
# Calculate the supply-demand ratio
# The supply-demand ratio is calculated by dividing the supply by the Pop_W_S (Population Weight Sum).
# This ratio indicates how well the supply meets the demand in each grid cell.
# A ratio greater than 1 indicates that the supply exceeds the demand, while a ratio less
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

In [ ]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [ ]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [ ]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [ ]:
origin_dest_acc

In [ ]:
# Find the maximum value of the Accessibility_standard
max(origin_dest_acc.Accessibility_standard)

In [ ]:
# Create a GeoDataFrame from the origin_dest_acc DataFrame
# Ensure the geometry is correctly created from the origin coordinates
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_closest", driver="GPKG")

## 4. Grouping by grid ID to prepare the final output file

In [ ]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_closest.gpkg')

In [ ]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [ ]:
# Remove duplicates based on the specified columns
# This ensures that each grid cell is unique in the results_grid DataFrame
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])

In [ ]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'drinking_water_deprivation_access.gpkg'
results_grid.to_file(output_gpkg_path, driver='GPKG')

In [ ]:
results_grid

### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  


In [ ]:
# Assign deprivation levels based on the Accessibility_standard values
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] >= 0, 'result'] = 2 #(high deprivation)
results_grid.loc[results_grid['Accessibility_standard'] > 0.0004202, 'result'] = 1 #(medium deprivation)
results_grid.loc[results_grid['Accessibility_standard'] > 0.0130206, 'result'] = 0 #(low deprivation)

### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [ ]:
#We set Dorayi Karama boundary as our focus area
from pyproj import CRS

def get_utm_crs(geo):
    lon, lat = geo.unary_union.centroid.x, geo.unary_union.centroid.y
    zone = int((lon + 180) // 6) + 1
    if lat >= 0:
        epsg_code = 32600 + zone  # Northern hemisphere
    else:
        epsg_code = 32700 + zone  # Southern hemisphere
    return CRS.from_epsg(epsg_code)


# 1. Read community boundary
dorayi_karama = gpd.read_file(data_inputs + "Dorayi_Karama.gpkg")

# Automatically choose the correct UTM CRS
utm_crs = get_utm_crs(dorayi_karama)

# Reproject boundary to UTM
dorayi_karama_utm = dorayi_karama.to_crs(utm_crs)

# Create 2km buffer
focus_area = dorayi_karama_utm.buffer(2000, join_style=2).unary_union


# Reproject grid to same UTM CRS
results_grid_utm = results_grid.to_crs(utm_crs)

# 3. Check intersection
results_grid_utm["focused"] = results_grid_utm.intersects(focus_area).astype(int)

# 4. Reproject back to original CRS (e.g., EPSG:4326)
results_grid = results_grid_utm.to_crs("EPSG:4326")

In [ ]:
# Remove rows where the result is -1 (no deprivation)
# This ensures that only relevant results are kept in the results_grid DataFrame
results_grid = results_grid.loc[results_grid['result'] != -1]

In [ ]:
# Rename columns for clarity
# This makes the column names more descriptive and easier to understand
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [ ]:
# Remove the 'geometry' column if it is not needed
results_grid

In [ ]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'drinking-water-access-class.gpkg'
results_grid.to_file(output_gpkg_path, layer='drinking-water-access-class', driver='GPKG')

In [ ]:
print(results_grid.columns)


In [ ]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid.drop(columns=['Accessibility_standard', 'grid_id', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)

In [ ]:
# Display the results table
results_table